# WESAD + SSL TimePatch Transformer

Incluye:
- cargar WESAD
- usar solo BVP, EDA, TEMP wrist
- convertir el problema a stress vs no-stress
- usar un TimePatch Transformer
- hacer self-supervised learning
- generar datos sintéticos parecidos a WESAD para aumentación
- evaluar con LOSO sobre sujetos reales

La idea central es comparar tres escenarios:
- solo supervisado
- SSL + supervisado
- SSL + datos sintéticos + supervisado

In [ ]:

!pip -q install numpy scipy scikit-learn matplotlib torch tqdm pandas

In [1]:

import os
import pickle
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


## Configurando la ruta del dataset

OJO: estructura esperada:

```text
WESAD/
  S2/S2.pkl
  S3/S3.pkl
  ...
```

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive/')

Mounted at /content/drive/


In [21]:

WESAD_ROOT = "/home/leoisidro/CICLOS/IX/TESIS/WESAD-PROCESAMIENTO/WESAD/WESAD"   # OJO actualicen al dataset
#la frecuencia objetivo:
TARGET_FS = 4
#la longitud de ventana:
WINDOW_SECONDS = 60
#el salto entre ventanas:
WINDOW_STRIDE_SECONDS = 30

USE_LABELS = {1, 2, 3}   # baseline, stress, amusement
STRESS_LABEL = 2
NOSTRESS_LABELS = {1, 3} #convirtiendo a binario

## cargando WESAD

In [22]:

#Cada señal de WESAD puede venir con distinta frecuencia.
#Esta función la remuestrea a una frecuencia común:
#todo se lleva a 4 Hz
#Esto sirve para que las tres señales tengan la misma longitud temporal y puedan entrar juntas al modelo.
def resample_1d(x, orig_fs, target_fs):
    x = np.asarray(x).squeeze()
    if orig_fs == target_fs:
        return x.astype(np.float32)
    duration = len(x) / float(orig_fs)
    target_len = max(2, int(round(duration * target_fs)))
    return signal.resample(x, target_len).astype(np.float32)

#Esta función normaliza cada señal de forma robusta usando:
#mediana
#rango intercuartílico

#Se usa esto en lugar de una normalización simple con media/desviación porque las señales fisiológicas suelen tener:
#picos
#ruido
#valores atípicos

#Así el modelo aprende patrones más estables entre sujetos.
def robust_norm(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isnan(x).any() else 0.0)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

#Esta función abre el .pkl de cada sujeto y extrae solo:

#BVP
#EDA
#TEMP
#label

#Luego:

#remuestrea cada señal a 4 Hz
#alinea las señales y etiquetas
#normaliza
#deja listo un arreglo temporal con 3 canales
def load_wesad_subject(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    wrist = data["signal"]["wrist"]
    labels = np.asarray(data["label"]).astype(int).squeeze()

    bvp = np.asarray(wrist["BVP"]).squeeze()
    eda = np.asarray(wrist["EDA"]).squeeze()
    temp = np.asarray(wrist["TEMP"]).squeeze()

    fs_bvp = 64
    fs_eda = 4
    fs_temp = 4
    fs_label = 700

    bvp = resample_1d(bvp, fs_bvp, TARGET_FS)
    eda = resample_1d(eda, fs_eda, TARGET_FS)
    temp = resample_1d(temp, fs_temp, TARGET_FS)
    labels = resample_1d(labels, fs_label, TARGET_FS).round().astype(int)

    L = min(len(bvp), len(eda), len(temp), len(labels))
    X = np.stack([
        robust_norm(bvp[:L]),
        robust_norm(eda[:L]),
        robust_norm(temp[:L]),
    ], axis=1)

    labels = labels[:L]
    keep = np.isin(labels, list(USE_LABELS))
    X = X[keep]
    labels = labels[keep]
    y = np.where(labels == STRESS_LABEL, 1, 0).astype(np.int64)
    return X, y

#Después, divide la señal continua en ventanas:

#cada ventana dura 60 s
#con stride de 30 s

#Entonces cada ventana tiene información temporal suficiente para captar respuesta fisiológica al estrés, y además se generan muchas muestras por sujeto.

#Cada ventana queda con forma aproximada:

#tiempo x canales
#o sea: T x 3
def segment_windows(X, y, window_seconds=60, stride_seconds=30, fs=4):
    win = int(window_seconds * fs)
    stride = int(stride_seconds * fs)
    Xw, yw = [], []

    for start in range(0, len(X) - win + 1, stride):
        end = start + win
        xw = X[start:end]
        yw_raw = y[start:end]
        frac = yw_raw.mean()
        if frac in [0.0, 1.0] or frac <= 0.2 or frac >= 0.8:
            Xw.append(xw.astype(np.float32))
            yw.append(int(frac >= 0.5))

    if len(Xw) == 0:
        return np.zeros((0, win, X.shape[1]), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    return np.stack(Xw), np.asarray(yw, dtype=np.int64)

#Se juntan todas las ventanas de todos los sujetos en tres variables:

#X: ventanas
#y: etiquetas binarias
#groups: identificador del sujeto

#groups es clave porque se usa para la evaluación Leave-One-Subject-Out.

def load_all_wesad(root):
    X_all, y_all, groups_all = [], [], []
    subject_dirs = sorted([d for d in os.listdir(root) if d.startswith("S")])

    for subj in subject_dirs:
        pkl_path = os.path.join(root, subj, f"{subj}.pkl")
        if not os.path.exists(pkl_path):
            print("Skipping:", pkl_path)
            continue
        X, y = load_wesad_subject(pkl_path)
        Xw, yw = segment_windows(X, y, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS, TARGET_FS)
        if len(Xw) == 0:
            continue
        sid = int(subj[1:])
        X_all.append(Xw)
        y_all.append(yw)
        groups_all.append(np.full(len(yw), sid))
        print(subj, Xw.shape, "stress ratio=", float(yw.mean()))

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    groups_all = np.concatenate(groups_all, axis=0)
    return X_all, y_all, groups_all

X, y, groups = load_all_wesad(WESAD_ROOT)
print("X:", X.shape, "y:", y.shape, "groups:", groups.shape)
print("Subjects:", np.unique(groups))
print("Stress ratio:", float(y.mean()))

S10 (73, 240, 3) stress ratio= 0.3013698630136986
S11 (71, 240, 3) stress ratio= 0.30985915492957744
S13 (72, 240, 3) stress ratio= 0.2916666666666667
S14 (71, 240, 3) stress ratio= 0.30985915492957744
S15 (72, 240, 3) stress ratio= 0.2916666666666667
S16 (71, 240, 3) stress ratio= 0.30985915492957744
S17 (73, 240, 3) stress ratio= 0.3013698630136986
S2 (67, 240, 3) stress ratio= 0.29850746268656714
S3 (68, 240, 3) stress ratio= 0.29411764705882354
S4 (70, 240, 3) stress ratio= 0.2857142857142857
S5 (70, 240, 3) stress ratio= 0.2714285714285714
S6 (70, 240, 3) stress ratio= 0.3
S7 (71, 240, 3) stress ratio= 0.28169014084507044
S8 (71, 240, 3) stress ratio= 0.29577464788732394
S9 (70, 240, 3) stress ratio= 0.3
X: (1060, 240, 3) y: (1060,) groups: (1060,)
Subjects: [ 2  3  4  5  6  7  8  9 10 11 13 14 15 16 17]
Stress ratio: 0.2962264150943396


In [23]:
# --- Loader desde processed_data (BVP + EDA + TEMP) para usar con TimePatchTransformer
import pandas as pd
import numpy as np

def robust_norm(x):
    x = np.asarray(x, dtype=np.float32)
    if np.isnan(x).any():
        x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isnan(x).any() else 0.0)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

def resample_1d(signal, target_len):
    signal = np.asarray(signal, dtype=np.float32)
    if len(signal) == target_len:
        return signal
    if len(signal) < 2:
        return np.full(target_len, float(signal[0]) if len(signal) else 0.0, dtype=np.float32)
    xp = np.linspace(0, 1, len(signal))
    x = np.linspace(0, 1, target_len)
    return np.interp(x, xp, signal).astype(np.float32)

def build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=240):
    # meta columns (incluye Participant para excluirla de features)
    meta_cols = {"Participant","Activity","PSS_BEFORE","PSS_AFTER","BASELINE_LABEL","TASK_LABEL","axis"}
    frames = {}

    for name, path in sensor_files.items():
        df = pd.read_csv(path, index_col=0)
        # Asegurar columna limpia `Participant` desde el índice (evita duplicados)
        participants_index = df.index.astype(str)
        df = df.reset_index(drop=True)
        df["Participant"] = participants_index

        # Comprobar existencia de label_col
        if label_col not in df.columns:
            raise ValueError(f"label_col '{label_col}' no está en {path}; columnas: {list(df.columns)[:10]}")

        # Columnas de señal: todas las columnas numéricas excepto las meta
        sig_cols = [c for c in df.columns if c not in meta_cols and c not in ("Participant","Activity")]
        frames[name] = df[["Participant","Activity"] + sig_cols + [label_col]].copy()

    # Añadir window index por participant+Activity para emparejar ventanas
    for k, df in frames.items():
        df["window_idx"] = df.groupby(["Participant","Activity"]).cumcount()
        frames[k] = df

    # Construir llave para merge: Participant + Activity + window_idx
    for k in frames:
        frames[k]["_key"] = frames[k]["Participant"].astype(str) + "||" + frames[k]["Activity"].astype(str) + "||" + frames[k]["window_idx"].astype(str)

    # Intersección de llaves comunes entre sensores
    common_keys = set.intersection(*[set(frames[k]["_key"].values) for k in frames])
    common_keys = sorted(common_keys)

    X_list = []
    y_list = []
    participants_list = []

    for key in common_keys:
        channel_data = []
        label_val = None
        part = None
        skip = False

        for name, df in frames.items():
            row = df[df["_key"] == key]
            if row.shape[0] != 1:
                skip = True
                break
            row = row.iloc[0]
            # Seleccionar columnas de señal (las guardadas en frames[name])
            sig = row[[c for c in df.columns if c not in {"Participant","Activity","window_idx","_key",label_col} and c not in meta_cols]].values.astype(np.float32)
            sig = robust_norm(sig)
            sig = resample_1d(sig, target_length)
            channel_data.append(sig)

            if label_val is None:
                # Etiquetar basado en Activity en lugar de PSS_AFTER
                activity = row['Activity']
                if activity == 'Baseline':
                    label_val = 0
                else:  # Stroop o Interview
                    label_val = 1
            if part is None:
                part = row["Participant"]

        if skip:
            continue

        # channel_data: lista de arrays (T,) por sensor -> apilar como (T, C)
        X_list.append(np.stack(channel_data, axis=1))
        y_list.append(label_val)
        participants_list.append(part)

    if len(X_list) == 0:
        return np.zeros((0, target_length, len(frames),), dtype=np.float32), np.zeros((0,), dtype=np.int64), np.zeros((0,), dtype=np.int64), []

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    groups = np.array([int(p.lstrip("S")) for p in participants_list], dtype=np.int64)

    return X, y, groups, participants_list

sensor_files = {
    "BVP": f"processed_data/BVP_{WINDOW_SECONDS}_seg.csv",
    "EDA": f"processed_data/EDA_{WINDOW_SECONDS}_seg.csv",
    "TEMP": f"processed_data/TEMP_{WINDOW_SECONDS}_seg.csv",
}

X_proc, y_proc, groups_proc, participants = build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=240)
print("Loaded processed_data -> X.shape:", X_proc.shape, "y.shape:", y_proc.shape, "groups.shape:", groups_proc.shape)

X1, y1, groups_1 = X_proc, y_proc, groups_proc

Loaded processed_data -> X.shape: (1616, 240, 3) y.shape: (1616,) groups.shape: (1616,)


## Datasets y aumentaciones

In [24]:

#Convierte X y y en un formato que PyTorch puede leer con DataLoader.

#Sirve tanto para:

#entrenamiento supervisado
#entrenamiento SSL
#validación
class WindowDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


#Esta función crea una versión modificada de una ventana.
#Las modificaciones son pequeñas, por ejemplo:

#añadir un poco de ruido
#escalar amplitud
#desplazar un poco en el tiempo
#enmascarar una parte pequeña de un canal

#Esto es fundamental para SSL, porque el modelo ve dos versiones del mismo ejemplo y aprende a reconocer que siguen representando el mismo estado fisiológico.
def weak_augment(x):
    x = x.clone()
    if random.random() < 0.8:
        x = x + 0.02 * torch.randn_like(x)
    if random.random() < 0.5:
        x = x * torch.empty((1, x.shape[1])).uniform_(0.9, 1.1)
    if random.random() < 0.4:
        shift = random.randint(-5, 5)
        x = torch.roll(x, shifts=shift, dims=0)
    if random.random() < 0.3:
        c = random.randrange(x.shape[1])
        start = random.randrange(0, max(1, x.shape[0] - 10))
        width = random.randrange(5, 15)
        x[start:start+width, c] = 0
    return x

#Genera pares (x1, x2) donde:

#x1 = una versión aumentada de la ventana
#x2 = otra versión aumentada de la misma ventana

#Esto se usa para aprendizaje contrastivo.
class SSLPairDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        return weak_augment(x), weak_augment(x)

## TimePatch Transformer

In [25]:
#En vez de meter toda la secuencia de golpe, la divide en pequeños bloques temporales llamados patches.

#Por ejemplo:

#si patch_len = 8
#entonces cada 8 pasos temporales se agrupan en un patch

#Cada patch se aplana y se proyecta a un vector latente.

#¿Por qué se hace esto?

#reduce longitud efectiva de la secuencia
#facilita que el Transformer trabaje mejor
#captura patrones locales antes de modelar dependencias largas
class PatchEmbed1D(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128):
        super().__init__()
        self.patch_len = patch_len
        self.proj = nn.Linear(in_ch * patch_len, emb_dim)

    def forward(self, x):
        B, T, C = x.shape
        P = self.patch_len
        T2 = (T // P) * P
        x = x[:, :T2, :].reshape(B, T2 // P, P * C)
        return self.proj(x)

#Después del patching, el modelo añade:

#un CLS token
#embeddings posicionales
#un TransformerEncoder

#Luego, el vector CLS se usa como resumen global de la ventana.

#El modelo tiene dos salidas implícitas:

#embedding para SSL
#clasificación para stress/no-stress

#En otras palabras, el Transformer aprende una representación temporal multimodal de la ventana.
class TimePatchTransformer(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128, depth=4, heads=4, num_classes=2):
        super().__init__()
        self.patch = PatchEmbed1D(in_ch, patch_len, emb_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_emb = nn.Parameter(torch.zeros(1, 512, emb_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=heads,
            dim_feedforward=emb_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(emb_dim)
        self.ssl_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, 64)
        )
        self.cls_head = nn.Linear(emb_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward_features(self, x):
        x = self.patch(x)
        B, N, D = x.shape
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb[:, :N+1, :]
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]

    def forward_ssl(self, x):
        z = self.forward_features(x)
        z = self.ssl_head(z)
        return F.normalize(z, dim=-1)

    def forward(self, x):
        z = self.forward_features(x)
        return self.cls_head(z)

## SSL y aprendizaje supervisado

In [26]:
#Esta es la pérdida contrastiva.

#La idea es:

#dos vistas aumentadas del mismo ejemplo deben quedar cerca en el espacio latente
#vistas de ejemplos distintos deben quedar lejos

#Eso obliga al modelo a aprender representaciones útiles sin usar etiquetas.
def nt_xent(z1, z2, temp=0.2):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=-1) / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.arange(B, device=z.device)
    targets = torch.cat([targets + B, targets], dim=0)
    return F.cross_entropy(sim, targets)

#Aquí se hace el preentrenamiento self-supervised.
#Pasos:

#se toman ventanas sin usar la etiqueta
#se generan dos vistas aumentadas
#el modelo produce embeddings para ambas
#se calcula la pérdida contrastiva
#se actualizan los pesos

#Resultado: el modelo aprende una representación inicial buena de las señales fisiológicas

def pretrain_ssl(model, X_unlabeled, epochs=6, batch_size=64, lr=1e-3):
    ds = SSLPairDataset(X_unlabeled)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    model.train()
    for ep in range(epochs):
        losses = []
        for x1, x2 in dl:
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            z1 = model.forward_ssl(x1)
            z2 = model.forward_ssl(x2)
            loss = nt_xent(z1, z2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        print(f"SSL epoch {ep+1}/{epochs}: {np.mean(losses):.4f}")
    return model

def evaluate_probs(y_true, probs):
    pred = (probs >= 0.5).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
    }
    try:
        out["auroc"] = roc_auc_score(y_true, probs)
    except Exception:
        out["auroc"] = np.nan
    return out

def finetune(model, X_train, y_train, X_val, y_val, epochs=8, batch_size=64, lr=2e-4):
    train_dl = DataLoader(WindowDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(WindowDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_f1 = -1

    for ep in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        probs_all, ys_all = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(DEVICE)
                probs = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
                probs_all.append(probs)
                ys_all.append(yb.numpy())

        probs_all = np.concatenate(probs_all)
        ys_all = np.concatenate(ys_all)
        metrics = evaluate_probs(ys_all, probs_all)
        print(f"FT epoch {ep+1}/{epochs}: loss={np.mean(train_losses):.4f}, val_f1={metrics['f1']:.4f}")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

def predict_probs(model, X_test, batch_size=128):
    dl = DataLoader(WindowDataset(X_test), batch_size=batch_size, shuffle=False)
    model.eval()
    probs = []
    with torch.no_grad():
        for xb in dl:
            xb = xb.to(DEVICE)
            p = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs)

## Experimentos LOSO sobre sujetos reales de WESAD

In [27]:
#Leave-One-Group-Out, donde el grupo es el sujeto.

#Eso significa:

#en cada fold, se deja un sujeto entero para test
#el entrenamiento se hace con los demás sujetos

#Esto evita “hacer trampa” mezclando ventanas del mismo sujeto en train y test.

#Dentro de cada fold se hace esto:
#a) Separar train y test
#X_train, y_train: todos los sujetos menos uno
#X_test, y_test: el sujeto dejado fuera

#b) Separar validación dentro del train
#Una pequeña parte del entrenamiento se usa como validación.

#c) Opción: añadir sintéticos
#Si use_synth=True:
#se generan ventanas sintéticas
#se añaden al conjunto de entrenamiento real

#d) Opción: hacer SSL
#Si use_ssl=True:
#primero se preentrena con las ventanas de entrenamiento
#luego se hace fine-tuning supervisado

#e) Evaluación
#Al final se calcula el rendimiento en el sujeto real no visto.
#Se reportan métricas como:
#accuracy
#F1
#balanced accuracy
#AUROC

def run_loso(X, y, groups, use_ssl=False, use_synth=False, ssl_epochs=6, ft_epochs=8, synth_ratio=0.5):
    logo = LeaveOneGroupOut()
    results = []

    for fold, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups), start=1):
        held_out = int(np.unique(groups[te_idx])[0])
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_test, y_test = X[te_idx], y[te_idx]

        n_val = max(8, int(0.15 * len(X_train)))
        perm = np.random.permutation(len(X_train))
        val_idx = perm[:n_val]
        tr2_idx = perm[n_val:]

        X_tr, y_tr = X_train[tr2_idx], y_train[tr2_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        if use_synth:
            Xs, ys = generate_synthetic_dataset(int(len(X_tr) * synth_ratio), class_balance=float(y_tr.mean()))
            X_tr = np.concatenate([X_tr, Xs], axis=0)
            y_tr = np.concatenate([y_tr, ys], axis=0)

        model = TimePatchTransformer(in_ch=3, patch_len=8, emb_dim=128, depth=4, heads=4, num_classes=2)

        if use_ssl:
            X_unlab = X_train.copy()
            if use_synth:
                Xu, _ = generate_synthetic_dataset(int(0.5 * len(X_train)), class_balance=float(y_train.mean()))
                X_unlab = np.concatenate([X_unlab, Xu], axis=0)
            model = pretrain_ssl(model, X_unlab, epochs=ssl_epochs)

        model = finetune(model, X_tr, y_tr, X_val, y_val, epochs=ft_epochs)
        probs = predict_probs(model, X_test)
        metrics = evaluate_probs(y_test, probs)
        metrics["subject"] = held_out
        results.append(metrics)
        print("Fold", fold, "subject", held_out, metrics)

    return results

def summarize(name, results, txt_path="resultados.txt", append=True):
    lines = ["", name]
    print("\n" + name)

    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        vals = np.array([r[key] for r in results], dtype=float)
        line = f"{key}: {np.nanmean(vals):.4f} ± {np.nanstd(vals):.4f}"
        lines.append(line)
        print(line)

    mode = "a" if append else "w"
    with open(txt_path, mode, encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

# Resultados WESAD

### Comparación de tres escenarios

Luego el notebook ejecuta tres experimentos completos:

1) Supervised only: Entrena directamente con etiquetas, sin SSL ni sintéticos Esto es baseline principal.

2) SSL + supervised: Primero hace self-supervised pretraining y luego fine-tuning supervisado. Esto sirve para comprobar si SSL ayuda.

3) SSL + synthetic + supervised:
Hace:
- pretraining SSL
- augmentación con ventanas sintéticas
- entrenamiento supervisado
Esto sirve para comprobar si además de SSL, los datos sintéticos mejoran aún más el resultado.

In [28]:
# 1) Supervised only
res_sup = run_loso(X, y, groups, use_ssl=False, use_synth=False, ft_epochs=8)
summarize("Supervised only", res_sup,f"resultados_Supervisado_WESAD_{WINDOW_SECONDS}seg")

FT epoch 1/8: loss=0.2930, val_f1=0.8000
FT epoch 2/8: loss=0.2450, val_f1=0.7733
FT epoch 3/8: loss=0.2389, val_f1=0.7838
FT epoch 4/8: loss=0.2028, val_f1=0.8286
FT epoch 5/8: loss=0.1882, val_f1=0.8493
FT epoch 6/8: loss=0.1966, val_f1=0.9114
FT epoch 7/8: loss=0.1396, val_f1=0.9091
FT epoch 8/8: loss=0.1061, val_f1=0.9351
Fold 1 subject 2 {'accuracy': 0.9850746268656716, 'f1': 0.975609756097561, 'balanced_accuracy': 0.9893617021276595, 'auroc': 1.0, 'subject': 2}
FT epoch 1/8: loss=0.2598, val_f1=0.7816
FT epoch 2/8: loss=0.2149, val_f1=0.8000
FT epoch 3/8: loss=0.1863, val_f1=0.7865
FT epoch 4/8: loss=0.1850, val_f1=0.8140
FT epoch 5/8: loss=0.1716, val_f1=0.8235
FT epoch 6/8: loss=0.1575, val_f1=0.8222
FT epoch 7/8: loss=0.1665, val_f1=0.8372
FT epoch 8/8: loss=0.1264, val_f1=0.8500
Fold 2 subject 3 {'accuracy': 0.7941176470588235, 'f1': 0.7083333333333334, 'balanced_accuracy': 0.8104166666666667, 'auroc': 0.828125, 'subject': 3}
FT epoch 1/8: loss=0.3559, val_f1=0.7949
FT epoch 

In [29]:

# 2) SSL + supervised
res_ssl = run_loso(X, y, groups, use_ssl=True, use_synth=False, ssl_epochs=6, ft_epochs=8)
summarize("SSL + supervised", res_ssl,f"resultados_SSL_WESAD_{WINDOW_SECONDS}seg")

SSL epoch 1/6: 2.8032
SSL epoch 2/6: 2.2356
SSL epoch 3/6: 2.0697
SSL epoch 4/6: 1.9658
SSL epoch 5/6: 1.8247
SSL epoch 6/6: 1.7413
FT epoch 1/8: loss=0.4130, val_f1=0.8444
FT epoch 2/8: loss=0.2377, val_f1=0.8421
FT epoch 3/8: loss=0.2136, val_f1=0.8723
FT epoch 4/8: loss=0.1822, val_f1=0.8864
FT epoch 5/8: loss=0.1608, val_f1=0.8864
FT epoch 6/8: loss=0.1385, val_f1=0.8913
FT epoch 7/8: loss=0.1082, val_f1=0.8696
FT epoch 8/8: loss=0.0944, val_f1=0.8791
Fold 1 subject 2 {'accuracy': 0.9701492537313433, 'f1': 0.9523809523809523, 'balanced_accuracy': 0.9787234042553192, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/6: 2.6981
SSL epoch 2/6: 2.1485
SSL epoch 3/6: 1.9206
SSL epoch 4/6: 1.8400
SSL epoch 5/6: 1.7322
SSL epoch 6/6: 1.6995
FT epoch 1/8: loss=0.2965, val_f1=0.8542
FT epoch 2/8: loss=0.2195, val_f1=0.8817
FT epoch 3/8: loss=0.2073, val_f1=0.8936
FT epoch 4/8: loss=0.1476, val_f1=0.8958
FT epoch 5/8: loss=0.1344, val_f1=0.9167
FT epoch 6/8: loss=0.1057, val_f1=0.9278
FT epoch 7/8: los

# Resultados STRESS ID

In [30]:
# 1) Supervised only
res_sup_1 = run_loso(X1, y1, groups_1, use_ssl=False, use_synth=False, ft_epochs=8)
summarize("Supervised only", res_sup_1,f"resultados_Supervisado_StressID_{WINDOW_SECONDS}seg")

FT epoch 1/8: loss=0.6964, val_f1=0.7124
FT epoch 2/8: loss=0.6693, val_f1=0.7759
FT epoch 3/8: loss=0.6619, val_f1=0.7855
FT epoch 4/8: loss=0.6484, val_f1=0.7768
FT epoch 5/8: loss=0.6325, val_f1=0.7788
FT epoch 6/8: loss=0.6344, val_f1=0.7859
FT epoch 7/8: loss=0.6391, val_f1=0.6885
FT epoch 8/8: loss=0.6305, val_f1=0.7839
Fold 1 subject 2 {'accuracy': 0.64, 'f1': 0.7352941176470589, 'balanced_accuracy': 0.5874363327674024, 'auroc': 0.7453310696095077, 'subject': 2}
FT epoch 1/8: loss=0.6949, val_f1=0.7298
FT epoch 2/8: loss=0.6743, val_f1=0.7389
FT epoch 3/8: loss=0.6582, val_f1=0.5328
FT epoch 4/8: loss=0.6588, val_f1=0.5844
FT epoch 5/8: loss=0.6432, val_f1=0.6993
FT epoch 6/8: loss=0.6447, val_f1=0.7435
FT epoch 7/8: loss=0.6439, val_f1=0.7380
FT epoch 8/8: loss=0.6227, val_f1=0.7301
Fold 2 subject 3 {'accuracy': 0.5714285714285714, 'f1': 0.6896551724137931, 'balanced_accuracy': 0.5176470588235295, 'auroc': 0.5576470588235294, 'subject': 3}
FT epoch 1/8: loss=0.6966, val_f1=0.72

In [31]:
res_ssl_1 = run_loso(X1, y1, groups_1, use_ssl=True, use_synth=False, ssl_epochs=6, ft_epochs=8)
summarize("SSL + supervised", res_ssl_1,f"resultados_SSL_StressID_{WINDOW_SECONDS}seg")

SSL epoch 1/6: 2.4338
SSL epoch 2/6: 1.8211
SSL epoch 3/6: 1.5842
SSL epoch 4/6: 1.5028
SSL epoch 5/6: 1.4588
SSL epoch 6/6: 1.4228
FT epoch 1/8: loss=0.6818, val_f1=0.6712
FT epoch 2/8: loss=0.6601, val_f1=0.7267
FT epoch 3/8: loss=0.6502, val_f1=0.7536
FT epoch 4/8: loss=0.6462, val_f1=0.6885
FT epoch 5/8: loss=0.6367, val_f1=0.7103
FT epoch 6/8: loss=0.6339, val_f1=0.7192
FT epoch 7/8: loss=0.6298, val_f1=0.6689
FT epoch 8/8: loss=0.6156, val_f1=0.6529
Fold 1 subject 2 {'accuracy': 0.7, 'f1': 0.7945205479452054, 'balanced_accuracy': 0.6256366723259762, 'auroc': 0.7843803056027165, 'subject': 2}
SSL epoch 1/6: 2.4091
SSL epoch 2/6: 1.7707
SSL epoch 3/6: 1.5749
SSL epoch 4/6: 1.4779
SSL epoch 5/6: 1.3940
SSL epoch 6/6: 1.4018
FT epoch 1/8: loss=0.6936, val_f1=0.7087
FT epoch 2/8: loss=0.6485, val_f1=0.7122
FT epoch 3/8: loss=0.6547, val_f1=0.6794
FT epoch 4/8: loss=0.6445, val_f1=0.6477
FT epoch 5/8: loss=0.6353, val_f1=0.6456
FT epoch 6/8: loss=0.6285, val_f1=0.7052
FT epoch 7/8: los